# goal: evaluate different manova computations

derive e and h from
- space and q naive (get_manova_naiva_sq)
- space naive (get_manova_naive_s)
- good way (hglm.regress.get_manova)

In [ ]:
import numpy as np
from tqdm import tqdm

def project(x, y, orth=False):
    p = x @ np.linalg.inv(x.T @ x) @ x.T
    if orth:
        # orthogonal
        n = y.shape[0]
        p = np.eye(n) - p
    return p @ y

def sample(seed=0, num_img=10, a=3, b=2, correlated_y=True, num_vox=1):    
    rng = np.random.default_rng(seed=seed)
    x = rng.standard_normal((num_img, a))
    y = rng.standard_normal((num_img * num_vox, b))

    if correlated_y:
        y_transform = rng.standard_normal(size=(b, b))
        y = y @ y_transform

    return x, y

In [ ]:
x, y = sample()

In [ ]:
contrast = np.ones(x.shape[1], dtype=bool)

a = (~contrast).sum(), contrast.size
to_sorted = np.eye(a[1])[np.argsort(contrast), :]
q, r = np.linalg.qr((x @ to_sorted), mode='complete')
q = q[:, :a[0]], q[:, a[0]: a[1]], q[:, a[1]:]

In [ ]:
import hglm

def get_manova_naive_sq(x, y, contrast=None):
    n = y.shape[0]
    
    # remove nuissance features
    if contrast is not None:
        y = project(x[:, ~contrast], y, orth=True)
        x = x[:, contrast]

    # compute e, error sum of squares
    error = project(x, y, orth=True)
    e = error.T @ error
    
    # compute h, variance explained by model (orthogonal to covariates)
    t = y.T @ y
    h = t - e
    
    return e, h

def get_manova_naive_s(x, y, contrast=None):
    if contrast is None:
        # every feature is of interest
        contrast = np.ones(x.shape[1], dtype=bool)
        
    a = (~contrast).sum(), contrast.size
    to_sorted = np.eye(a[1])[np.argsort(contrast), :]
    q, r = np.linalg.qr((x @ to_sorted), mode='complete')
    q = q[:, :a[0]], q[:, a[0]: a[1]], q[:, a[1]:]

    # compute e, error sum of squares
    e = y.T @ q[2] @ q[2].T @ y
    e = e.T @ e
    
    # compute h, variance explained by model (orthogonal to covariates)
    h = y.T @ q[1] @ q[1].T @ y
    h = h.T @ h
    
    return e, h

In [ ]:
x, y = sample()
e0, h0 = get_manova_naive_sq(x, y)
e1, h1 = get_manova_naive_s(x, y)
    
#     np.testing.assert_allclose(e0, e1)
#     np.testing.assert_allclose(h0, h1)

In [ ]:
h0 / h1

# wilks is chi2 distributed

sanity check: under the null hypothesis wilks lambda is approximately chi square distributed

In [ ]:
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

def get_wilks(e, h):   
    return np.linalg.det(e) / np.linalg.det(e + h)

def wilks_to_chi2(wilks, a, b, n):
    scale =  -(n - 0.5 * (a + b + 1))
    return scale * np.log(wilks)

def chi2_to_wilks(chi2, a, b, n):
    scale =  -(n - 0.5 * (a + b + 1))
    return np.exp(chi2 / scale)

In [ ]:
n = 1000
num_img = 10
a = 3
b = 2
contrast=np.array([True, True, False])

# observed
chi2_observed = []
for x, y in sample(n=n, num_img=num_img, a=a, b=b):
    e, h = get_manova_naive_sq(x, y, contrast=contrast) 
    wilks = get_wilks(e, h)
    chi2 = wilks_to_chi2(wilks, a=sum(contrast), b=b, n=n)
    chi2_observed.append(chi2)
    
# predicted
df = sum(contrast) * b
x = np.linspace(0, max(chi2_observed), 100)
pdf = stats.chi2.pdf(x, df)

In [ ]:
# plot
plt.plot(x, pdf, 'r-', label=f'Theoretical Chi-Sq(df={int(df)})')
plt.hist(chi2_observed, bins=30, density=True, alpha=0.6, color='g', label='Observed Chi2')
plt.xlabel('Chi2')
plt.legend()

plt.tight_layout()
plt.show()

# sanity check 2
permutation testing on real data also follows chi2

next: compute wilks using formulas above (is it different?)

In [ ]:
import numpy as np

# additive white gaussian noise
seed = 0
shape = 5, 5, 5
a, b, num_img = 2, 3, 10
n_perm = 10000

# sample y
rng = np.random.default_rng(seed=seed)
num_vox = np.prod(shape)
y = rng.standard_normal(size=(b, num_img, num_vox))

# induce correlated y features
y_transform = rng.standard_normal(size=(b, b))
y = np.einsum('bnr,bc->cnr', y, y_transform)

# build experiment (sample x features)
mask_idx = hglm.mask.get_mask_idx(np.ones(shape))
exp = hglm.experiment.ExperimentImageOnly(y=y, mask_idx=mask_idx)
exp = exp.sample_x(a=a, seed=seed)

In [ ]:
# check that e and h are the same (whats the deal with contrast?)
e, h = hglm.experiment.regress.get_manova(x=exp.x, y=exp.y, contrast=exp.contrast)

In [ ]:
_y = np.hstack([exp.y[:, :, i] for i in range(num_vox)]).T
_x = np.hstack([exp.x for i in range(num_vox)]).T
_x.shape, _y.shape

In [ ]:
_e, _h = get_manova(_x, _y, contrast=exp.contrast)

In [ ]:
_e / e, _h/h

In [ ]:
# from tqdm import tqdm

# # observed
# chi2_observed = list()
# for seed in tqdm(range(n_perm)):
#     _exp = exp.permute(perm_idx=seed, block_exchange=False)
    
#     e, h = hglm.experiment.regress.get_manova(x=_exp.x, y=_exp.y, contrast=_exp.contrast)
#     wilks = np.linalg.det(e) / np.linalg.det(e + h)
#     chi2 = wilks_to_chi2(wilks, a=exp.contrast.sum(), b=b, n=num_img * np.prod(shape))
#     chi2_observed.append(chi2)
    
# # predicted
# df = exp.contrast.sum() * b
# x = np.linspace(0, max(chi2_observed), 100)
# pdf = stats.chi2.pdf(x, df)

In [ ]:
# # plot
# plt.hist(chi2_observed, bins=30, weights=np.ones(n_perm) / n_perm, alpha=0.6, color='g', label='Observed Chi2')
# plt.plot(x, pdf, 'r-', label=f'Theoretical Chi-Sq(df={int(df)})')
# plt.xlabel('Chi2')
# plt.legend()

# plt.tight_layout()
# plt.show()